# 🧾 Week 5 hands-on: Answer quality and evidence

## Learning Objectives
In this notebook, you will learn:
1. **The answer cases and writer v1** - load 24 cases with fixed evidence and see what an extractive writer says
2. **Seven evidence checks** - faithfulness, citation support, correctness, relevance, abstention, stale and restricted sources
3. **A report by slice** - find the answers that are faithful to their evidence and still wrong
4. **Access-aware retrieval** - rerun with filtered search and separate retrieval failures from generation failures
5. **Breaking and fixing** - make the writer abstain, then work through two debugging drills

## Where this fits
This is the runnable companion to `week-05-answer-quality-and-evidence-handson.md`, using the same code as `lab.py` (a test keeps the two identical). Read `week-05-answer-quality-and-evidence-theory.md` first. Week 4 measured the search. This week measures the answer written from it, and Week 6 compares two versions of both.

## Prerequisites
- `week-05-answer-quality-and-evidence-theory.md`, and Week 4
- The course environment from `new_course/setup.md`, selected as this notebook's kernel
- No API keys for the offline parts. The optional Ragas cell in Part 6 needs `requirements-integrations.txt` and `OPENAI_API_KEY`, and costs money

---
## 📦 Step 1: Environment Setup

The setup cell finds `new_course/`, imports the shared writer and retrievers, and names the answer cases.

In [ ]:
# ============================================================================
# ENVIRONMENT SETUP: Paths, the writer and the retrievers
# ============================================================================
import json
import sys
from collections import defaultdict
from dataclasses import dataclass
from pathlib import Path

# --- Find new_course/ from wherever Jupyter started (the notebook's folder or new_course/)
COURSE = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "shared" / "support_agent.py").exists())
sys.path.insert(0, str(COURSE))

from shared.answerer import ABSTAIN_TEXT, Answer, answer_v1, overlap, split_sentences  # noqa: E402
from shared.retrieval import Chunk, Retriever, chunk_documents, load_documents  # noqa: E402

CASES = COURSE / "datasets" / "answers" / "answer-cases-v1.jsonl"

print(f"✅ Environment ready: {CASES.relative_to(COURSE)}")

---
## 📄 Part 1: The Answer Cases and Writer v1

Each case fixes the **evidence**, the chunks the writer receives, so a failure can only come from the writer. Writer v1 quotes the two best-matching sentences and cites the chunk each came from. It has three deliberate bugs: it quotes any chunk, never abstains, and ignores conflicts.

In [ ]:
# ============================================================================
# CASES AND WRITER: Run writer v1 on the 60-day case
# ============================================================================
def read_jsonl(path: Path) -> list[dict]:
    return [json.loads(line) for line in path.read_text().splitlines() if line.strip()]


def norm(text: str) -> str:
    return " ".join(text.lower().split())


cases = read_jsonl(CASES)
by_id = {c.chunk_id: c for c in chunk_documents(load_documents())}
case = next(c for c in cases if c["case_id"] == "a-009")
answer = answer_v1(case["question"], [by_id[cid] for cid in case["retrieved"]])
print(f"📋 {len(cases)} cases. {case['case_id']}: {case['question']}")
for sentence, cited in answer.sentences:
    print(f"🤖 {sentence}  [{cited}]")
print(f"📄 Reference: {case['reference_answer']}")

Both sentences are quoted from the evidence and correctly cited. One of them is the superseded 60-day rule.

---
## 🔍 Part 2: Seven Evidence Checks

Each check reads the answer against a different reference point: the evidence (faithfulness, citations), the reference facts (correctness), the gold sections (relevance), the expected behavior (abstention) and the chunk metadata (stale and restricted sources).

### Key Concepts:
- **Faithfulness**: every claim is in the evidence. It says nothing about whether the evidence was right
- **Forbidden fact**: a phrase a correct answer must never contain. It applies even when the case expects abstention
- **Restricted source**: internal text in a customer answer. Critical

In [ ]:
# ============================================================================
# CHECKS: The result type, the severities and the seven checks
# ============================================================================
CHECK_SEVERITY = {
    "faithfulness": "major", "citation_support": "major", "correctness": "major",
    "relevance": "minor", "abstention": "major", "stale_source": "major", "restricted_source": "critical",
}


@dataclass
class CheckResult:
    check: str
    passed: bool
    reason: str

    @property
    def severity(self) -> str:
        return CHECK_SEVERITY[self.check]


def check_faithfulness(case: dict, chunks: list[Chunk], answer: Answer) -> CheckResult:
    """Every answer sentence must appear in some chunk the writer was given."""
    evidence = " ".join(norm(c.text) for c in chunks)
    unsupported = [s for s, _cid in answer.sentences if norm(s) not in evidence]
    if answer.abstained:
        return CheckResult("faithfulness", True, "abstained: nothing to support")
    reason = "every sentence is in the evidence" if not unsupported else f"unsupported: {unsupported}"
    return CheckResult("faithfulness", not unsupported, reason)


def check_citation_support(case: dict, chunks: list[Chunk], answer: Answer) -> CheckResult:
    """Each sentence's cited chunk must itself contain the sentence."""
    by_id = {c.chunk_id: c for c in chunks}
    wrong = [cid for s, cid in answer.sentences
             if cid is None or cid not in by_id or norm(s) not in norm(by_id[cid].text)]
    reason = "every citation supports its sentence" if not wrong else f"citations that do not support: {wrong}"
    return CheckResult("citation_support", not wrong, reason)


def check_correctness(case: dict, chunks: list[Chunk], answer: Answer) -> CheckResult:
    """The answer states the required facts and none of the forbidden ones.

    Required facts apply only when the case expects an answer. Forbidden facts apply
    always: an abstain case can still leak a fact it must not state.
    """
    text = norm(answer.text)
    required = case["must_include"] if case["expected_behavior"] == "answer" else []
    missing = [f for f in required if norm(f) not in text]
    present = [f for f in case["must_not_include"] if norm(f) in text]
    ok = not missing and not present
    reason = "required facts present" if ok else f"missing {missing}, forbidden {present}"
    return CheckResult("correctness", ok, reason)


def check_relevance(case: dict, chunks: list[Chunk], answer: Answer) -> CheckResult:
    """The answer draws on at least one section that answers the question."""
    if case["expected_behavior"] == "abstain":
        return CheckResult("relevance", True, "abstain case: no gold section")
    cited_sections = set(answer.citations())
    ok = bool(cited_sections & set(case["gold_sections"]))
    return CheckResult("relevance", ok, "cites a gold section" if ok else f"cites none of {case['gold_sections']}")


def check_abstention(case: dict, chunks: list[Chunk], answer: Answer) -> CheckResult:
    """Abstain exactly when the case says the evidence cannot support an answer."""
    want = case["expected_behavior"] == "abstain"
    ok = answer.abstained == want
    reason = ("abstained as expected" if ok and want else "answered as expected" if ok
              else "answered when it should abstain" if want else "abstained when it should answer")
    return CheckResult("abstention", ok, reason)


def check_stale_source(case: dict, chunks: list[Chunk], answer: Answer) -> CheckResult:
    """No sentence may be cited to a superseded policy."""
    by_id = {c.chunk_id: c for c in chunks}
    stale = [cid for cid in answer.citations() if cid in by_id and by_id[cid].status != "current"]
    return CheckResult("stale_source", not stale, "no superseded sources" if not stale else f"cites superseded {stale}")


def check_restricted_source(case: dict, chunks: list[Chunk], answer: Answer) -> CheckResult:
    """No sentence may come from an internal document. This is the access check."""
    by_id = {c.chunk_id: c for c in chunks}
    leaked = [cid for cid in answer.citations() if cid in by_id and by_id[cid].audience != "public"]
    return CheckResult("restricted_source", not leaked,
                       "no internal sources" if not leaked else f"quotes internal {leaked}")


CHECKS = [check_faithfulness, check_citation_support, check_correctness, check_relevance,
          check_abstention, check_stale_source, check_restricted_source]


print(f"✅ Defined {len(CHECKS)} checks: {[c.__name__.removeprefix('check_') for c in CHECKS]}")

---
## 🧮 Part 3: A Report by Slice

The runner gives every case its evidence, runs the writer and applies all seven checks. The report counts passes per slice and lists the two groups that matter most: critical failures, and answers that are faithful but incorrect.

In [ ]:
# ============================================================================
# REPORT: Writer v1 on fixed evidence
# ============================================================================
def evidence_for(case: dict, by_id: dict[str, Chunk], retriever: Retriever | None) -> list[Chunk]:
    """The fixed evidence from the case, or a fresh filtered hybrid search."""
    if retriever is None:
        return [by_id[cid] for cid in case["retrieved"]]
    return retriever.search(case["question"], 3, filtered=True)


def run_cases(writer=answer_v1, retrieval: str = "fixed", cases: list[dict] | None = None) -> list[dict]:
    cases = cases or read_jsonl(CASES)
    chunks = chunk_documents(load_documents())
    by_id = {c.chunk_id: c for c in chunks}
    retriever = Retriever(chunks, "hybrid") if retrieval == "filtered" else None
    rows = []
    for case in cases:
        evidence = evidence_for(case, by_id, retriever)
        answer = writer(case["question"], evidence)
        results = [check(case, evidence, answer) for check in CHECKS]
        rows.append({"case": case, "evidence": evidence, "answer": answer, "results": results})
    return rows


def print_report(rows: list[dict], title: str) -> None:
    print(title)
    names = [c.__name__.removeprefix("check_") for c in CHECKS]
    short = {"faithfulness": "faith", "citation_support": "cite", "correctness": "correct",
             "relevance": "relev", "abstention": "abstain", "stale_source": "stale",
             "restricted_source": "restrict"}
    print(f"{'slice (cases)':<20}" + "".join(f"{short[n]:>9}" for n in names) + f"{'all':>7}")
    groups: dict[str, list[dict]] = defaultdict(list)
    for r in rows:
        groups[r["case"]["slice"]].append(r)
    for name, group in [*groups.items(), ("overall", rows)]:
        cells = "".join(f"{sum(x['results'][i].passed for x in group):>6}/{len(group):<2}" for i in range(len(names)))
        all_ok = sum(all(res.passed for res in x["results"]) for x in group)
        print(f"{name + f' ({len(group)})':<20}{cells}{all_ok:>4}/{len(group)}")
    critical = [r["case"]["case_id"] for r in rows
                if any(not res.passed and res.severity == "critical" for res in r["results"])]
    faithful_but_wrong = [r["case"]["case_id"] for r in rows
                          if r["results"][0].passed and not r["results"][2].passed]
    print(f"\nCritical failures (internal text in an answer): {critical}")
    print(f"Faithful to the evidence but incorrect: {faithful_but_wrong}")


def print_case(case_id: str, retrieval: str) -> None:
    rows = [r for r in run_cases(retrieval=retrieval) if r["case"]["case_id"] == case_id]
    if not rows:
        sys.exit(f"unknown case ID: {case_id}")
    r = rows[0]
    print(f"{case_id} ({r['case']['slice']}): {r['case']['question']}")
    print(f"evidence: {[c.chunk_id for c in r['evidence']]}")
    for s, cid in r["answer"].sentences:
        print(f"  answer: {s}  [{cid}]")
    print(f"reference: {r['case']['reference_answer']}")
    for res in r["results"]:
        print(f"  {'PASS' if res.passed else 'FAIL'}  {res.check:<18} [{res.severity}] {res.reason}")


print_report(run_cases(), "Writer v1, fixed evidence from each case:")

Nine answers are faithful and incorrect. Faithfulness alone would have passed all of them.

---
## ⚖️ Part 4: Access-Aware Retrieval

Now the writer gets a filtered hybrid search instead of the fixed evidence. Failures that move with the evidence are retrieval failures. Failures that stay are generation failures.

> **Key Insight**: The filter takes critical failures from 4 to 0, and the missing slice stays at 0 of 4.

In [ ]:
# ============================================================================
# FILTERED: The same cases with access-aware retrieval
# ============================================================================
print_report(run_cases(retrieval="filtered"), "Writer v1, evidence from a filtered hybrid search, top 3:")
print()
print_case("a-021", "fixed")
print()
print_case("a-021", "filtered")

---
## 🐛 Part 5: Breaking and Fixing

### 5.1 ✍️ Your writer
Exercise 3: change the fallback so the writer abstains with `ABSTAIN_TEXT` when no sentence scores 1.5 or more. The version below does that. Edit it, then rerun to see which checks change.

In [ ]:
# ============================================================================
# YOUR WRITER: Abstain instead of guessing
# ============================================================================
def writer_that_abstains(question, chunks, max_sentences=2):
    scored = [(overlap(question, s), rank, s, c.chunk_id) for rank, c in enumerate(chunks) for s in split_sentences(c)]
    scored.sort(key=lambda t: (-t[0], t[1]))
    picked = [t for t in scored[:max_sentences] if t[0] >= 1.5]
    if not picked:
        return Answer(ABSTAIN_TEXT, [], abstained=True, writer="mine")
    sentences = [(s, cid) for _score, _rank, s, cid in picked]
    return Answer(" ".join(s for s, _cid in sentences), sentences, writer="mine")


print_report(run_cases(writer=writer_that_abstains), "Abstaining writer, fixed evidence:")

### 5.2 🐛 Drill 1: the citations point at the wrong rule
Swap case a-001's two citations. Faithfulness cannot see the swap, and citation support can.

In [ ]:
# ============================================================================
# DRILL 1: Swapped citations
# ============================================================================
row = next(r for r in run_cases() if r["case"]["case_id"] == "a-001")
(s1, c1), (s2, c2) = row["answer"].sentences
swapped = Answer(row["answer"].text, [(s1, c2), (s2, c1)], writer="swapped")
print(f"✅ {check_faithfulness(row['case'], row['evidence'], swapped)}")
print(f"❌ {check_citation_support(row['case'], row['evidence'], swapped)}")

### 5.3 🐛 Drill 2: a correct answer fails correctness
"3-5 business days" and "3 to 5 business days" are the same fact. Phrase matching disagrees.

In [ ]:
# ============================================================================
# DRILL 2: Same fact, different format
# ============================================================================
right_but_reformatted = Answer("Card refunds appear within 3-5 business days.",
                               [("Card refunds appear within 3-5 business days.", "payments-policy-2026-01#P2")])
print(f"❌ {check_correctness(row['case'], row['evidence'], right_but_reformatted)}")
normalized = Answer(right_but_reformatted.text.replace("3-5", "3 to 5"), right_but_reformatted.sentences)
print(f"✅ After normalizing ranges: {check_correctness(row['case'], row['evidence'], normalized)}")

---
## 💳 Part 6: Optional Ragas Scores (paid)

This cell is tagged `skip-execution` because it calls a paid model judge: on the order of 150 to 200 judge calls, estimated from how the metrics work. To run it by hand, install `requirements-integrations.txt` (it pins `openai` 2.54.0 for Ragas 0.4.3), set `OPENAI_API_KEY`, change `RUN_LIVE` to `True` and run the cell. The pins and APIs were checked in an isolated environment, and the command has not been run against a model yet.

In [ ]:
# ============================================================================
# RAGAS: Model-judged scores beside the local checks (costs money)
# ============================================================================
def ragas_live(model: str = "gpt-4o-mini") -> None:
    """Score the same answers with Ragas 0.4.3. Needs requirements-integrations.txt and OPENAI_API_KEY.

    Four metrics per case: about 24 x 4 judge calls plus embeddings for answer relevancy.
    """
    import asyncio
    import os
    if not os.environ.get("OPENAI_API_KEY"):
        sys.exit("OPENAI_API_KEY is not set. ragas --live is optional; `report` runs offline.")
    from openai import AsyncOpenAI
    from ragas.embeddings import OpenAIEmbeddings
    from ragas.llms.base import llm_factory
    from ragas.metrics.collections import AnswerRelevancy, ContextPrecision, ContextRecall, Faithfulness

    client = AsyncOpenAI()
    judge = llm_factory(model, client=client)
    metrics = {
        "faithfulness": Faithfulness(llm=judge),
        "answer_relevancy": AnswerRelevancy(llm=judge, embeddings=OpenAIEmbeddings(
            client=client, model="text-embedding-3-small")),
        "context_precision": ContextPrecision(llm=judge),
        "context_recall": ContextRecall(llm=judge),
    }

    async def score(row: dict) -> dict:
        case, contexts = row["case"], [c.text for c in row["evidence"]]
        q, a, ref = case["question"], row["answer"].text, case["reference_answer"]
        return {
            "faithfulness": (await metrics["faithfulness"].ascore(
                user_input=q, response=a, retrieved_contexts=contexts)).value,
            "answer_relevancy": (await metrics["answer_relevancy"].ascore(user_input=q, response=a)).value,
            "context_precision": (await metrics["context_precision"].ascore(
                user_input=q, reference=ref, retrieved_contexts=contexts)).value,
            "context_recall": (await metrics["context_recall"].ascore(
                user_input=q, retrieved_contexts=contexts, reference=ref)).value,
        }

    rows = run_cases()
    print(f"{'case':<7}{'slice':<14}{'local: faith':>13}{'local: correct':>15}"
          f"{'ragas: faith':>13}{'ragas: relev':>13}{'ragas: ctx P':>13}{'ragas: ctx R':>13}")
    for row in rows:
        s = asyncio.run(score(row))
        local = row["results"]
        print(f"{row['case']['case_id']:<7}{row['case']['slice']:<14}"
              f"{'pass' if local[0].passed else 'FAIL':>13}{'pass' if local[2].passed else 'FAIL':>15}"
              f"{s['faithfulness']:>13.2f}{s['answer_relevancy']:>13.2f}"
              f"{s['context_precision']:>13.2f}{s['context_recall']:>13.2f}")
    print(f"\nRagas columns are model-judged by {model} and vary between runs. Local columns are deterministic.")


RUN_LIVE = False  # set to True to spend money on Ragas judge calls
if RUN_LIVE:
    ragas_live()
else:
    print("⚠️ Ragas skipped. Set RUN_LIVE = True to run it.")

---
## 📝 Summary

In this notebook, we learned:

### 1. The answer cases and writer v1
- Fixing the evidence per case makes every failure a writer failure, which isolates generation from retrieval.

### 2. Seven evidence checks
- Each check reads a different reference point. Faithfulness reads the evidence, correctness the policy facts, and the source checks the metadata.

### 3. A report by slice
- Nine answers are faithful to their evidence and still state the wrong policy.

### 4. Access-aware retrieval
- The filter removes every stale and restricted source. It cannot make the writer abstain.

### 5. Breaking and fixing
- Abstention fixes the missing slice. Swapped citations and reformatted facts each fool a different check.

### Next Steps
- **`week-06-rag-regression-comparisons-handson.ipynb`**: compares version 1 with a candidate version 2 and decides whether to ship it.